In [1]:
import bw2analyzer as ba
import bw2data as bd
import bw2calc as bc
import bw2io as bi
import matrix_utils as mu
import bw_processing as bp
import time
import wurst
import os
from pathlib import Path
import copy
import pandas as pd

In [2]:
bd.projects.set_current('Acetylene production')
list(bd.databases)

['ecoinvent-3.10-biosphere',
 'ecoinvent-3.10-cutoff',
 'ecoinvent-3.10-cutoff-Base-2020',
 'ecoinvent-3.10-cutoff-Base-2050',
 'ecoinvent-3.10-cutoff-RCP26-2050',
 'ecoinvent-3.10-cutoff-RCP19-2050',
 'acetylene-production-Base-2020',
 'acetylene-production-Base-2050',
 'acetylene-production-RCP19-2050',
 'acetylene-production-RCP26-2050',
 'acetylene-production']

In [3]:
db = bd.Database('acetylene-production')

In [4]:
acts = [act for act in db if 'production' in act['name']
        and 'wind electricity' not in act['name']
        and 'calcium carbide production' not in act['name']]
len(acts)

12

In [5]:
methods = [("IPCC 2021", "climate change", "GWP 100a, incl. H and bio CO2")]
len(methods)

1

In [6]:
mc_df = pd.DataFrame()

for act in acts:
    lca= bc.LCA({act: 1}, methods[0], use_distributions = True)
    lca.lci()
    lca.lcia()
    results = [lca.score for _ in zip(lca, range(500))]

    if mc_df.empty:
        mc_df = pd.DataFrame(results, columns = [act['name']])
    else:
        mc_df[act['name']] = results

In [7]:
mc_df

,"acetylene production, natural gas, partial oxidation","vinyl chloride monomer production, fossil ethylene","acetylene production, biochar, calcium carbide","acetylene production, biomethane, cold plasma","acetylene production, natural gas, cold plasma","acetylene production, coal, calcium carbide","vinyl chloride monomer production, bio acetylene","vinyl chloride monomer production, bio ethylene","acetylene production, biomethane, hot plasma","acetylene production, natural gas, hot plasma","acetylene production, biomethane, partial oxidation","vinyl chloride monomer production, fossil acetylene"
0,8.022386,2.414866,3.254579,5.766668,10.037234,13.983809,2.535710,0.929564,6.488145,11.597262,-5.673661,5.839017
1,9.863540,2.381458,2.782766,6.420465,10.686683,13.582844,1.744523,0.897261,6.982411,11.512603,-5.663185,5.624586
2,9.044084,2.675215,4.873014,6.232072,12.103038,14.053218,2.270433,0.903995,6.593106,11.225513,-6.185300,6.927276
3,8.030861,2.942600,3.776139,5.105805,11.256530,16.425278,1.605379,1.028090,6.939645,11.920317,-5.763877,6.053243
4,8.518814,2.053880,4.600057,6.104137,10.102321,14.679931,2.052518,0.940547,6.776748,12.641158,-5.260417,6.271751
...,...,...,...,...,...,...,...,...,...,...,...,...
495,7.843673,2.604629,4.727531,6.329563,10.205511,15.328667,2.269539,0.942581,7.332900,11.924548,-5.605131,5.862401
496,8.201293,2.742233,5.440316,5.750464,9.753417,13.645848,1.929368,0.737674,6.879835,12.997817,-5.527707,5.988274
497,8.423657,2.345169,4.205879,6.195438,10.747364,14.386172,1.467735,0.841720,7.514807,11.515631,-5.829499,6.238856
498,8.620658,3.091960,5.591860,5.818069,10.458161,13.823973,2.503653,0.899742,6.214042,11.674319,-5.853079,5.990280


In [8]:
mc_df.to_excel(os.path.join('..', 'results', 'mc_results.xlsx'))